⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Migration: TRG_EMP_LOAD.txt\n",
        "\n",
        "**Conversion Timestamp:** 2024-07-30 12:00:00\n",
        "\n",
        "**Description:** This notebook loads data directly into the `trg_emp` table from the `employees` source table. It converts a simple Oracle INSERT statement to Databricks Spark SQL."
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"ETL Job Type\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\", \"Datasource Number ID\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\", \"ETL Process ID\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\", \"ODI Session Number\")\n",
        "dbutils.widgets.text(\"ETL_LAST_EXTRACT_TIME\", \"1900-01-01 00:00:00\", \"Last Extract Time (yyyy-MM-dd HH:mm:ss)\")\n",
        "dbutils.widgets.text(\"ETL_CURRENT_EXTRACT_TIME\", \"2099-12-31 23:59:59\", \"Current Extract Time (yyyy-MM-dd HH:mm:ss)\")"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## ETL Parameters"
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS\n",
        "SELECT\n",
        "  '${ETL_JOB_TYPE}' AS etl_job_type,\n",
        "  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,\n",
        "  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,\n",
        "  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,\n",
        "  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,\n",
        "  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "code",
      "source": [
        "display(spark.sql(\"SELECT * FROM v_etl_parameters\"))"
      ],
      "metadata": {
        "collapsed": false
      }
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Target Table Setup and Load"
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO in {10}: Placeholder for potential initialization\n",
        "-- SCEN_TASK_NO in {20}: Placeholder for potential pre-processing\n",
        "\n",
        "-- Ensure target table exists (example DDL, adjust as needed based on actual source DDL)\n",
        "-- Assuming standard HR schema types:\n",
        "-- NUMBER(6) -> BIGINT\n",
        "-- VARCHAR2(20) -> STRING\n",
        "-- DATE -> TIMESTAMP\n",
        "-- NUMBER(8,2) -> DECIMAL(8,2)\n",
        "-- NUMBER(2,2) -> DECIMAL(2,2)\n",
        "CREATE TABLE IF NOT EXISTS workspace.hr.trg_emp (\n",
        "  employee_id    BIGINT,\n",
        "  first_name     STRING,\n",
        "  last_name      STRING,\n",
        "  email          STRING,\n",
        "  phone_number   STRING,\n",
        "  hire_date      TIMESTAMP,\n",
        "  job_id         STRING,\n",
        "  salary         DECIMAL(8,2),\n",
        "  commission_pct DECIMAL(2,2),\n",
        "  manager_id     BIGINT,\n",
        "  department_id  BIGINT\n",
        ") USING DELTA;"
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO in {30}: Insert into target table\n",
        "\n",
        "-- Delete existing records from target if this is a full refresh or truncate is desired\n",
        "-- For a simple INSERT, we assume the target is empty or managed externally.\n",
        "-- If this was intended as a full refresh, a TRUNCATE/DELETE would precede this INSERT.\n",
        "\n",
        "INSERT INTO workspace.hr.trg_emp\n",
        "(\n",
        "  employee_id,\n",
        "  first_name,\n",
        "  last_name,\n",
        "  email,\n",
        "  phone_number,\n",
        "  hire_date,\n",
        "  job_id,\n",
        "  salary,\n",
        "  commission_pct,\n",
        "  manager_id,\n",
        "  department_id\n",
        ")\n",
        "SELECT\n",
        "  employees.employee_id,\n",
        "  employees.first_name,\n",
        "  employees.last_name,\n",
        "  employees.email,\n",
        "  employees.phone_number,\n",
        "  employees.hire_date,\n",
        "  employees.job_id,\n",
        "  employees.salary,\n",
        "  employees.commission_pct,\n",
        "  employees.manager_id,\n",
        "  employees.department_id\n",
        "FROM\n",
        "  workspace.hr.employees AS employees;"
      ],
      "metadata": {}
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Validation"
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) AS total_records_inserted\n",
        "FROM workspace.hr.trg_emp;"
      ],
      "metadata": {}
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT\n",
        "  employee_id,\n",
        "  first_name,\n",
        "  last_name,\n",
        "  hire_date,\n",
        "  salary\n",
        "FROM workspace.hr.trg_emp\n",
        "ORDER BY employee_id\n",
        "LIMIT 10;"
      ],
      "metadata": {}
    },
    {
      "cell_type": "markdown",
      "source": [
        "## Conversion Notes and Manual Actions Required\n",
        "\n",
        "1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.`. Please ensure `workspace.hr` refers to the correct catalog and schema in your Databricks environment.\n",
        "2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Delta Lake.\n",
        "3.  **Data Types:** An example DDL for `workspace.hr.trg_emp` has been provided based on common Oracle HR schema types. You **must review and verify** this DDL against the actual source table's Oracle DDL to ensure correct Spark SQL data type mapping (e.g., `NUMBER` to `BIGINT` or `DECIMAL`, `DATE` to `TIMESTAMP`).\n",
        "4.  **ETL Parameters:** Placeholder widgets for common ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, etc.) and corresponding temporary views (`v_etl_parameters`) have been created as per the standard notebook template. These are not directly used in the provided simple `INSERT` statement but are included for completeness and adherence to the template.\n",
        "5.  **Insert Behavior:** The original SQL was a direct `INSERT`. This conversion assumes the target table `workspace.hr.trg_emp` is either empty before execution or that existing data should be duplicated/managed by external processes. If this `INSERT` was part of a "full load" strategy (where the target table is truncated/deleted first), you would need to add an explicit `TRUNCATE TABLE` or `DELETE FROM` statement before the `INSERT`."
      ],
      "metadata": {}
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "python",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.9.18"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}